# PRD v3 risk level: simulator upgrade, retraining and Thai benchmark

**Purpose.** Retrain the AGAP-CKD risk-level model on an upgraded synthetic population, set the Low / Medium / High cut-offs, apply the guideline floor, and benchmark it against the Thai simplified CKD score, as specified in `docs/AGAP-CKD-PRD-v3.md`, ADR 0001 and ADR 0002.

**Safety framing.** Every person here is synthetic. The model estimates the chance of an **abnormal kidney screen** (one measured eGFR < 60 mL/min/1.73 m² or urine ACR ≥ 30 mg/g). A risk level is a prompt to get a kidney test, **never a diagnosis**: chronic kidney disease needs abnormality persisting over 3 months (KDIGO 2024).

**Reproducibility.** All logic lives in `src/agap/simulate/risk_check.py` and `src/agap/triage.py` (tested in `tests/test_triage.py`); this notebook only calls it. `make triage` produces the same artifacts. Every parameter is in `config/params.yaml` and classified in `config/parameter_register.yaml`.

## 1. Methodology

### 1.1 Study design
The model is developed on a **synthetic development cohort in which everyone has a kidney test**, mirroring the population surveys behind the Thai simplified score (Thakkinstian et al., 2011), SCORED (Bang et al., 2007) and the Long An study (Tran et al., 2017). This avoids the selection problem of learning only from people who chose to be tested. The cohort contains people eligible for the risk check: anyone already told something was wrong with their kidneys is routed to care and excluded (PRD v3 item 13).

| Split | Size | Seed |
|---|---|---|
| Training | 10,000 eligible adults | master seed 2026, component 101 |
| Validation (sets the cut-offs) | 5,000 eligible adults | component 102 |
| Test (held out; benchmark reported here) | 5,000 eligible adults | component 105 |
| Prevalence calibration | 200,000 adults (population level) | component 103 |
| Prevalence sensitivity | 5,000 per scenario | component 104 |

### 1.2 Population
Adults 18+ are drawn by the PRD v3 age groups with **Laguna shares from the WorldPop 2020 age-sex rasters** (census-based modelled counts; the PSA census tables are not in the repository): 18–39 59.0%, 40–49 18.4%, 50–59 12.1%, 60–69 6.8%, 70+ 3.8%; 51.4% female. Exact ages are uniform within groups (70+ over 70–95).

### 1.3 True traits and outcome generation
Diabetes and hypertension use the repository's existing age-dependent generators (diabetes calibrated to FNRI 2023 measured high fasting glucose). New traits: heart disease or stroke (age-dependent), kidney stones, family history, heavy outdoor work, BMI, and medicine use among people aware of diabetes or hypertension. A synthetic community heat exposure is also drawn.

The underlying kidney state is drawn from a logistic model:

$$\text{logit}\,P(\text{damage}) = \beta_0 + \sum_k \log(\text{OR}_k)\,x_k$$

| Effect | OR used | Class | Evidence |
|---|---|---|---|
| Age 40–49 and 50–59 (vs 18–39) | 1.8 | derived | Thakkinstian 2011, OR 1.8 for 40–59 |
| Age 60–69 | 4.1 | source-supported | Thakkinstian 2011 |
| Age 70+ | 8.3 | source-supported | Thakkinstian 2011 |
| Diabetes | 2.5 | source-supported | Thakkinstian 2011 |
| Hypertension | 2.3 | source-supported | Thakkinstian 2011 |
| Kidney stones | 2.8 | source-supported | Thakkinstian 2011 |
| Female | 1.2 | assumed | SCORED item (Bang 2007); OR not retrieved |
| Heart disease or stroke | 1.75 | assumed | SCORED item; PRD v3 range 1.5–2 |
| Family history | 1.75 | assumed | QKidney predictor (Hippisley-Cox 2010) |
| Overweight (BMI 23–27.4) | 1.1 | derived | Garofalo 2017: RR 1.06 (low eGFR), 1.24 (albuminuria), not significant |
| Obese (BMI ≥ 27.5) | 1.4 | derived | Garofalo 2017: RR 1.28 (low eGFR), 1.51 (albuminuria) |
| Stopped medicine | 1.2 | derived | Chang 2015: HR 1.11 for ESRD |
| Heavy outdoor work | 1.3 | assumed | CKDu mechanism (Johnson 2019); no effect size exists |
| Community heat, per SD | 1.2 | assumed | outcome generation only; never a model input (ADR 0001) |

Where a relative risk or hazard ratio is used as an odds ratio, the approximation is loose at about 13% prevalence; those effects are registered as derived, not source-supported.

### 1.4 Labs, measurement and the outcome
Underlying eGFR and urine ACR are drawn consistent with the kidney state from **truncated continuous** distributions (no pile-up at the thresholds). The **day-0 screen is one measurement** with within-person variability (creatinine CV 5%, uACR CV 30%), so a single result can flag a healthy person or miss damage near a threshold. The outcome is that measured result, computed with CKD-EPI 2021 (Inker et al., 2021).

### 1.5 Prevalence calibration
The intercept β₀ is solved by Monte Carlo bisection on a fixed-seed population of 200,000 so the **measured** abnormal-screen share, at population level before routing, is **12.76%** (Tran et al., 2017: 260 of 2,037 adults in Long An, Vietnam, ACR ≥ 30 mg/g or MDRD eGFR < 60). No population-based Philippine estimate was found.

### 1.6 Self-reported answers
Self-report misses undiagnosed disease: in the Utrecht Health Project, self-reported diabetes had 58.9% sensitivity (Molenaar et al., 2007), and 61.6% of hypertensive older Filipinos knew their status (Abalos et al., 2024). The simulator therefore makes 60% of true cases aware; aware cases answer Yes, unaware cases answer "I don't know" or No. Every item allows "I don't know".

### 1.7 Model
- **Inputs (18):** age group, female sex, diabetes, high blood pressure, heart disease or stroke, kidney stones, family history, heavy outdoor work, BMI category (Asian cut-offs 23 and 27.5; WHO Expert Consultation, 2004).
- **"I don't know":** for diabetes and high blood pressure it enters as **Yes** (PRD v3), and a "High" BP reading counts as high blood pressure. For the other items it is its own category, not an imputed value.
- **Excluded by design:** labs, hidden traits, kidney status and heat (ADR 0001), enforced by `check_allowed_features` against `models.forbidden_features`.
- **Fit:** L2-penalized logistic regression (scikit-learn, C = 1.0). Logistic regression is the default for clinical prediction; machine learning has shown no general performance benefit over it (Christodoulou et al., 2019).
- **Uncertainty:** 95% Wald intervals from the inverse penalized Hessian (the penalty applies to coefficients, not the intercept).

### 1.8 Risk levels and guideline floor
Cut-offs are set on the validation cohort (PRD v3) and the benchmark is reported on a separate held-out test cohort, so the tiers are not scored on the data that tuned them:
- **Medium-or-High:** the highest threshold whose Medium-or-High group catches **≥ 85%** of abnormal screens.
- **High:** the lowest threshold above it whose High group has an abnormal rate **≥ 2× prevalence**, with at least 30 abnormal screens.

The **guideline floor** (ADR 0002) is applied after the model: diabetes or high blood pressure (explicit Yes or a "High" reading) → at least Medium; both, or either with age ≥ 60 → High. "I don't know" raises the model's probability but does **not** trigger the floor (PRD v3 open question; implemented as specified).

### 1.9 Benchmark and evaluation
- **Thai simplified score** (Thakkinstian et al., 2011): age < 40 = 1, 40–59 = 2, 60–69 = 4, ≥ 70 = 8; diabetes 3 vs 1; hypertension 2 vs 1; kidney stones 3 vs 1; range 4–16. It uses the same diabetes and blood-pressure answer mapping as AGAP, so the comparison isolates the scoring method.
- **Three arms:** Thai score; AGAP model alone; AGAP model + guideline floor.
- **Metrics:** AUROC, sensitivity (share of abnormal screens in Medium/High), specificity (share of normal screens in Low), the share and abnormal rate in each level, and decision-curve net benefit (Vickers & Elkin, 2006).
- **Prevalence sensitivity:** the frozen model and cut-offs re-evaluated at 8%, 12.8% and 18% population prevalence.

## 2. Deviations from the specification

The specification for this run was checked against the cited papers. Where they disagreed, the literature was followed and the change is recorded here.

| Specification | What the literature check found | What this notebook does |
|---|---|---|
| Age effects "50–59 OR ≈ 2.0, 60–69 OR ≈ 3.0, 70+ OR ≈ 4.0" from SCORED | Those are SCORED **points**, not odds ratios (Bang et al., 2007); SCORED's odds ratios were not retrieved | Uses the verified Thai odds ratios: 40–59 1.8, 60–69 4.1, ≥ 70 8.3 (Thakkinstian 2011, Table 2) |
| Age group 40–49 has no stated OR | The Thai paper pools 40–59 | Applies 1.8 to both 40–49 and 50–59 |
| Thai points with reference categories = 1, range 4–16 | The paper's Table 2 lists reference categories as 0, while its text and risk bands use the 4–16 scheme | Uses the 4–16 scheme, the only one consistent with the published bands. This is **not** a constant shift: only reference categories gain a point, which narrows each risk factor's gap and can reorder people (e.g. age 70+ alone: 8 → 11; age 40–59 with diabetes, hypertension and stones: 10 → 10). The Table 2 scheme's AUROC is reported as a sensitivity check. |
| Three Thai tiers (Low 4–5, Medium 6–8, High ≥ 9) | The paper has four bands: 4–5, 6–8, 9–11, ≥ 12 | Collapses 9–11 and ≥ 12 into High; a team choice, labelled as such |
| Obesity "RR ≈ 1.3 for eGFR < 60, 1.5 for uACR ≥ 30"; overweight RR ≈ 1.1 | Confirmed: obesity RR 1.28 and 1.51; overweight 1.06 and 1.24, neither significant (Garofalo et al., 2017) | One OR on the combined outcome: obese 1.4, overweight 1.1 (derived) |
| Stopped medicine ×1.1–1.5 (Roy 2013, Chang 2015) | Chang 2015 HR 1.11 confirmed via a secondary table; Roy 2013 HR 0.67 not retrievable | OR 1.2, registered as derived from Chang only |
| Female OR ≈ 1.2 (Bang 2007) | Female sex is a SCORED item, but its OR was not retrieved | OR 1.2, registered as **assumed** |
| Laguna age structure from the PSA census | PSA tables are not in the repository | WorldPop 2020 age-sex rasters for Laguna (census-based modelled counts) |
| Prevalence 12.76% (Tran 2017) | Confirmed; that study used MDRD on a screening sample | Same target; the simulator uses CKD-EPI 2021; difference noted |
| "Triage tier proportions remain stable" across prevalences | Abnormal rates per level must shift with prevalence | Reports the shift rather than claiming stability |

**Benchmark caveat.** Because the simulator's age, diabetes, hypertension and kidney-stone effects come from the Thai paper, the Thai score is nearly correctly specified on its four items. The comparison is therefore tilted **toward** the Thai score, which is conservative for AGAP.

In [1]:
import json, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd

from agap.config import ROOT, load_config
from agap.experiment import fingerprint
from agap.simulate.risk_check import simulate_population
from agap.triage import (FEATURES, benchmark, coefficient_table, encode_features, guideline_floor, model_levels,
                         prevalence_sensitivity, rule_net_benefit, save_artifacts, train_and_evaluate)

pd.set_option("display.width", 200, "display.max_columns", 30, "display.precision", 3)
cfg = load_config()
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
print(f"config fingerprint {fingerprint(cfg)} · git commit {commit} · master seed {cfg['seed']}")

config fingerprint 3fa0a298f6ea · git commit de2888e · master seed 2026


## 3. Synthetic cohorts and prevalence check

In [2]:
rc = cfg["simulator"]["risk_check"]
population = simulate_population(cfg, 200_000, np.random.default_rng(cfg["seed"] * 1000 + 103))
result = train_and_evaluate(cfg)
train, validation = result["train"], result["validation"]
pd.DataFrame([
    dict(group="Population (before routing)", n=len(population), measured_abnormal_screen=population.y_true.mean(),
         underlying_kidney_damage=population.kidney_damage_true.mean(),
         routed_already_told=(population.kidney_status == "Told something was wrong").mean()),
    dict(group="Training cohort (eligible)", n=len(train), measured_abnormal_screen=train.y_true.mean(),
         underlying_kidney_damage=train.kidney_damage_true.mean(), routed_already_told=0.0),
    dict(group="Validation cohort (eligible)", n=len(validation), measured_abnormal_screen=validation.y_true.mean(),
         underlying_kidney_damage=validation.kidney_damage_true.mean(), routed_already_told=0.0),
]).assign(target=rc["target_prevalence"])

,group,n,measured_abnormal_screen,underlying_kidney_damage,routed_already_told,target
0,Population (before routing),200000,0.128,0.114,0.011,0.128
1,Training cohort (eligible),10000,0.122,0.108,0.000,0.128
2,Validation cohort (eligible),5000,0.115,0.101,0.000,0.128


In [3]:
# A single measurement disagrees with the underlying kidney state for some people (Section 1.4).
pd.crosstab(population.kidney_damage_true, population.y_true, rownames=["underlying damage"],
            colnames=["measured abnormal screen"], normalize="all").round(4)

measured abnormal screen,False,True
underlying damage,,
False,0.869,0.017
True,0.004,0.111


In [4]:
# Answers vs truth: self-report misses undiagnosed diabetes and hypertension (Section 1.6).
summary = []
for item, truth in (("diabetes", "dm_true"), ("high_blood_pressure", "htn_true")):
    shares = population[item].value_counts(normalize=True)
    summary.append(dict(item=item, true_prevalence=population[truth].mean(), answered_yes=shares.get("Yes", 0),
                        answered_dont_know=shares.get("I don't know", 0),
                        yes_among_true_cases=(population.loc[population[truth], item] == "Yes").mean()))
pd.DataFrame(summary)

,item,true_prevalence,answered_yes,answered_dont_know,yes_among_true_cases
0,diabetes,0.103,0.061,0.090,0.590
1,high_blood_pressure,0.186,0.112,0.095,0.601


In [5]:
validation.groupby("age_group").y_true.agg(["size", "mean"]).rename(columns={"size": "n", "mean": "abnormal_screen_rate"})

,n,abnormal_screen_rate
age_group,,
18-39,3010,0.071
40-49,888,0.119
50-59,581,0.131
60-69,346,0.254
70+,175,0.503


## 4. Features and leakage check

In [6]:
X_train, X_validation = result["X_train"], result["X_validation"]
print(len(FEATURES), "features:", ", ".join(FEATURES))
forbidden = set(cfg["models"]["forbidden_features"])
assert not forbidden & set(X_train.columns), "forbidden feature in the model matrix"
X_validation.mean().rename("share of validation cohort").to_frame().T

18 features: age_40_49, age_50_59, age_60_69, age_70_plus, female, diabetes_model, hypertension_model, heart_disease_stroke_yes, heart_disease_stroke_unknown, kidney_stones_yes, kidney_stones_unknown, family_history_yes, family_history_unknown, heavy_outdoor_work_yes, heavy_outdoor_work_unknown, bmi_overweight, bmi_obese, bmi_unknown


,age_40_49,age_50_59,age_60_69,age_70_plus,female,diabetes_model,hypertension_model,heart_disease_stroke_yes,heart_disease_stroke_unknown,kidney_stones_yes,kidney_stones_unknown,family_history_yes,family_history_unknown,heavy_outdoor_work_yes,heavy_outdoor_work_unknown,bmi_overweight,bmi_obese,bmi_unknown
share of validation cohort,0.178,0.116,0.069,0.035,0.518,0.15,0.219,0.037,0.045,0.043,0.049,0.177,0.053,0.287,0.05,0.316,0.174,0.156


## 5. Fitted model: odds ratios with 95% Wald intervals

In [7]:
coefficients = result["coefficients"]
odds = rc["odds_ratios"]
generating = {"age_40_49": odds["age_40_49"], "age_50_59": odds["age_50_59"], "age_60_69": odds["age_60_69"],
              "age_70_plus": odds["age_70_plus"], "female": odds["female"], "diabetes_model": odds["diabetes"],
              "hypertension_model": odds["hypertension"], "heart_disease_stroke_yes": odds["heart_disease_stroke"],
              "kidney_stones_yes": odds["kidney_stones"], "family_history_yes": odds["family_history"],
              "heavy_outdoor_work_yes": odds["heavy_outdoor_work"], "bmi_overweight": odds["overweight"],
              "bmi_obese": odds["obese"]}
coefficients.assign(simulator_or=coefficients.term.map(generating))[
    ["term", "coefficient", "se", "odds_ratio", "or_low_95", "or_high_95", "simulator_or"]]

,term,coefficient,se,odds_ratio,or_low_95,or_high_95,simulator_or
0,intercept,-3.017,0.086,0.049,0.041,0.058,NaN
1,age_40_49,0.443,0.086,1.558,1.315,1.845,1.80
2,age_50_59,0.447,0.096,1.563,1.294,1.888,1.80
3,age_60_69,1.227,0.102,3.410,2.792,4.165,4.10
4,age_70_plus,1.934,0.131,6.917,5.348,8.946,8.30
5,female,0.153,0.064,1.165,1.027,1.321,1.20
6,diabetes_model,0.354,0.082,1.425,1.215,1.672,2.50
7,hypertension_model,0.665,0.069,1.945,1.698,2.228,2.30
8,heart_disease_stroke_yes,0.517,0.132,1.678,1.296,2.171,1.75
9,heart_disease_stroke_unknown,-0.071,0.150,0.931,0.693,1.251,NaN


Fitted odds ratios sit below the simulator's values for diabetes and hypertension. That is expected:
- **Unaware cases:** the model sees *answers*, not true disease, and unaware cases often answer No.
- **"I don't know" → Yes:** this mixes some healthy people into the Yes group.
- **Noisy outcome:** the measured outcome is noisy.

The model omits the community heat effect by design (ADR 0001).

## 6. Validation cut-offs

In [8]:
cutoffs = result["cutoffs"]
pd.Series(cutoffs).to_frame("value")

,value
t_medium,0.071
t_high,0.121
prevalence,0.115
medium_or_high_sensitivity,0.852
high_abnormal_rate,0.229
high_target_rate,0.229


## 7. Benchmark: Thai score vs AGAP model vs model + guideline floor (held-out test cohort)

In [9]:
table = result["benchmark"]
table[["arm", "auroc", "sensitivity", "specificity", "referred_share", "share_low", "abnormal_rate_low",
       "share_medium", "abnormal_rate_medium", "share_high", "abnormal_rate_high"]]

,arm,auroc,sensitivity,specificity,referred_share,share_low,abnormal_rate_low,share_medium,abnormal_rate_medium,share_high,abnormal_rate_high
0,Thai simplified score,0.691,0.585,0.709,0.327,0.673,0.077,0.272,0.178,0.055,0.440
1,AGAP model alone,0.711,0.863,0.335,0.689,0.311,0.055,0.372,0.086,0.318,0.237
2,AGAP model + guideline floor,0.711,0.865,0.331,0.693,0.307,0.055,0.376,0.086,0.318,0.237


In [10]:
from sklearn.metrics import roc_auc_score
from agap.triage import thai_points_table2
test = result["test"]
print(f"test prevalence {test.y_true.mean():.3f}; 2x target {2 * test.y_true.mean():.3f}; "
      f"High abnormal rate on test {table.loc[1, 'abnormal_rate_high']:.3f} (rule set on validation: {cutoffs['high_abnormal_rate']:.3f})")
print(f"Thai AUROC, 4-16 scheme {roc_auc_score(test.y_true, result['scored']['points']):.3f}; "
      f"Table 2 (reference 0) scheme {roc_auc_score(test.y_true, thai_points_table2(test)):.3f}")
print("Same metrics on the validation cohort that set the cut-offs (optimistic):")
result["validation_benchmark"][["arm", "auroc", "sensitivity", "specificity", "abnormal_rate_high"]]

test prevalence 0.124; 2x target 0.248; High abnormal rate on test 0.237 (rule set on validation: 0.229)
Thai AUROC, 4-16 scheme 0.691; Table 2 (reference 0) scheme 0.694
Same metrics on the validation cohort that set the cut-offs (optimistic):


,arm,auroc,sensitivity,specificity,abnormal_rate_high
0,Thai simplified score,0.703,0.583,0.713,0.471
1,AGAP model alone,0.722,0.852,0.339,0.229
2,AGAP model + guideline floor,0.722,0.853,0.334,0.229


In [11]:
scored = result["scored"]
floor = scored["floor"]
moved = (np.maximum(scored["levels"], floor) != scored["levels"])
print(f"guideline floor raised {moved.sum()} of {len(moved)} test-cohort people "
      f"({moved.mean():.1%}); abnormal-screen rate among them {scored['y'][moved].mean():.1%}")

guideline floor raised 19 of 5000 test-cohort people (0.4%); abnormal-screen rate among them 5.3%


## 8. Decision-curve analysis

In [12]:
sensitivity_table = prevalence_sensitivity(result["model"], cutoffs, cfg)
outputs = ROOT / "outputs"
frozen = save_artifacts(result, cfg, sensitivity_table, outputs, commit)
dca = ROOT / ".agents/skills/ckd-health-data-scientist/scripts/decision_curve.py"
for name in ("model", "thai"):
    print(f"=== {name}: probabilities ({'logistic map of Thai points on the test cohort' if name == 'thai' else 'AGAP model'})")
    out = subprocess.run([sys.executable, str(dca), "--file", str(outputs / f"triage_dca_{name}.csv"),
                          "--thresholds", "0.05,0.10,0.15,0.20"], capture_output=True, text=True)
    print(out.stdout or out.stderr)

=== model: probabilities (AGAP model)

Decision Curve Analysis (DCA) - Clinical Net Benefit:
Threshold  | NB (Model)  | NB (All)  | NB (None) | Screens/1k  | Cases/1k  | Avoided Screens/1k
-----------------------------------------------------------------------------------------------
0.050      | 0.0792      | 0.0781    | 0.0000    | 935.2       | 122.0     | 20.8              
0.100      | 0.0476      | 0.0269    | 0.0000    | 415.4       | 84.4      | 186.6             
0.150      | 0.0316      | -0.0304   | 0.0000    | 215.4       | 59.2      | 351.3             
0.200      | 0.0209      | -0.0948   | 0.0000    | 128.4       | 42.4      | 462.6             

Interpretation: A model is clinically useful at a given threshold if its Net Benefit (Model)
exceeds both Net Benefit (All) and Net Benefit (None = 0.0).

=== thai: probabilities (logistic map of Thai points on the test cohort)

Decision Curve Analysis (DCA) - Clinical Net Benefit:
Threshold  | NB (Model)  | NB (All)  | NB (None

In [13]:
# Net benefit of each arm's referral rule (refer = Medium or High), at fixed threshold probabilities.
y = scored["y"]
rules = {"Thai simplified score": scored["points"] >= 6, "AGAP model alone": scored["levels"] >= 1,
         "AGAP model + guideline floor": np.maximum(scored["levels"], floor) >= 1,
         "Refer everyone": np.ones_like(y, bool)}
pd.DataFrame({name: {f"pt={t:.2f}": rule_net_benefit(y, refer, t) for t in (0.05, 0.10, 0.15, 0.20)}
              for name, refer in rules.items()}).T

,pt=0.05,pt=0.10,pt=0.15,pt=0.20
Thai simplified score,0.059,0.044,0.028,0.009
AGAP model alone,0.077,0.043,0.004,-0.038
AGAP model + guideline floor,0.077,0.042,0.004,-0.039
Refer everyone,0.078,0.027,-0.030,-0.095


## 9. Prevalence sensitivity (frozen model and cut-offs)

In [14]:
sensitivity_table[["target_prevalence", "cohort_prevalence", "arm", "auroc", "sensitivity", "referred_share",
                   "abnormal_rate_low", "abnormal_rate_medium", "abnormal_rate_high"]]

,target_prevalence,cohort_prevalence,arm,auroc,sensitivity,referred_share,abnormal_rate_low,abnormal_rate_medium,abnormal_rate_high
0,0.080,0.073,Thai simplified score,0.671,0.563,0.321,0.047,0.088,0.322
1,0.080,0.073,AGAP model alone,0.684,0.838,0.688,0.038,0.057,0.128
2,0.080,0.073,AGAP model + guideline floor,0.684,0.838,0.691,0.038,0.056,0.128
3,0.128,0.109,Thai simplified score,0.700,0.590,0.320,0.066,0.157,0.422
4,0.128,0.109,AGAP model alone,0.704,0.861,0.697,0.050,0.080,0.203
5,0.128,0.109,AGAP model + guideline floor,0.704,0.865,0.701,0.049,0.081,0.203
6,0.180,0.170,Thai simplified score,0.690,0.555,0.326,0.112,0.230,0.550
7,0.180,0.170,AGAP model alone,0.713,0.863,0.689,0.075,0.124,0.318
8,0.180,0.170,AGAP model + guideline floor,0.713,0.866,0.691,0.074,0.125,0.318


## 10. Verification

In [15]:
checks = [
    ["leakage scanner", sys.executable, ".agents/skills/ckd-health-data-scientist/scripts/leakage_scanner.py",
     "--file", "outputs/validation_features.csv"],
    ["parameter register", sys.executable, ".agents/skills/ckd-health-researcher/scripts/parameter_register_validator.py"],
    ["focused tests", sys.executable, "-m", "pytest", "-q", "--color=no", "tests/test_triage.py", "tests/test_leakage.py",
     "tests/test_parameter_register.py"],
]
for name, *command in checks:
    out = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
    print(f"--- {name} (exit {out.returncode})\n{(out.stdout + out.stderr).strip().splitlines()[-1]}")

--- leakage scanner (exit 0)
[PASS] All 18 features passed leakage audit. No forbidden features detected.
--- parameter register (exit 0)
[PASS] Parameter Register Audit Successful: All parameters classified with evidence.


--- focused tests (exit 0)
16 passed in 2.11s


## 11. Interpretation and limitations

- **Fallback rule.** The cut-offs were found, High exceeds Medium, and every check passed, so the run is technically usable. The PRD's 3:00 AM fallback deadline has passed, so whether the deck switches to these numbers is a team decision. If it does, freeze this commit and run and never mix in older numbers. If a future run raises `ValueError` in `set_cutoffs`, keep the frozen model with the guideline floor and present the new questions as "collected for the next model version".
- **Discrimination (held-out test cohort).** AUROC is 0.711 for the AGAP model and 0.691 for the Thai score (0.694 under the paper's Table 2 points). The benchmark is tilted toward the Thai score (Section 2), and both use the same answers for diabetes and blood pressure. On the validation cohort that set the cut-offs, the figures are 0.722 and 0.703, which is optimistic.
- **Sensitivity is an operating point, not a better model.** AGAP catches 86% of abnormal screens against the Thai bands' 59%, but only because it refers 69% of people against 33%. AUROC and the continuous decision curve are the fair comparison. The referral rule is the team's trade-off.
- **Net benefit.** As a continuous score, the AGAP model has higher net benefit than the logistic-mapped Thai score at threshold probabilities 0.05–0.15 (0.048 vs 0.044 at 0.10); the Thai score is slightly higher at 0.20. The Thai map is fitted on the same test cohort, which favours it slightly. As fixed referral rules, the AGAP Medium cut-off (a predicted chance of about 7%) beats the Thai bands at 0.05 and falls behind from 0.10. The 85% catch target is defensible only if testing a person with roughly a 7% chance of an abnormal screen is judged worthwhile. That is a decision for the team and clinicians.
- **High tier out of sample.** The High cut-off was set as the lowest threshold reaching 2× prevalence on validation (22.9% against 22.9%). On the held-out test cohort, High's abnormal rate is 23.7% against a 2× target of 24.8%, and it also misses at 8%, 12.8% and 18% prevalence. The 2× property holds only on the data that set it. A margin, or a larger validation cohort, would make it hold out of sample.
- **Guideline floor.** The floor changed 19 of 5,000 test-cohort levels (0.4%), because most people who explicitly report diabetes or hypertension already exceed the Medium cut-off. It still guarantees the guideline rule in every case.
- **Prevalence.** The share referred is stable across 8–18% prevalence, but the abnormal rate inside each level moves with prevalence. Results must be quoted at a stated prevalence.
- **What the numbers are.** All results are simulation results under registered, partly assumed effects. The model recovers effects that were put into the simulator; it does not show how strong each factor is in Laguna. Real validation needs Philippine data with measured eGFR and urine ACR.
- **Wording.** Results are risk levels and abnormal kidney screens, never diagnoses, "CKD cases" or "positives".

## References

Abalos, J. B., et al. (2024). Prevalence, awareness, treatment, and control of hypertension among older adults in the Philippines. *The Journals of Gerontology: Series A, 79*(2), glad155. https://doi.org/10.1093/gerona/glad155

Bang, H., Vupputuri, S., Shoham, D. A., Klemmer, P. J., Falk, R. J., Mazumdar, M., Gipson, D., Colindres, R. E., & Kshirsagar, A. V. (2007). SCreening for Occult REnal Disease (SCORED): A simple prediction model for chronic kidney disease. *Archives of Internal Medicine, 167*(4), 374–381.

Chang, P. Y., Chien, L. N., Lin, Y. F., Chiou, H. Y., & Chiu, W. T. (2015). Nonadherence of oral antihyperglycemic medication will increase risk of end-stage renal disease. *Medicine, 94*(47), e2051.

Christodoulou, E., Ma, J., Collins, G. S., Steyerberg, E. W., Verbakel, J. Y., & Van Calster, B. (2019). A systematic review shows no performance benefit of machine learning over logistic regression for clinical prediction models. *Journal of Clinical Epidemiology, 110*, 12–22.

Garofalo, C., Borrelli, S., Minutolo, R., Chiodini, P., De Nicola, L., & Conte, G. (2017). A systematic review and meta-analysis suggests obesity predicts onset of chronic kidney disease in the general population. *Kidney International, 91*(5), 1224–1235. https://doi.org/10.1016/j.kint.2016.12.013

Inker, L. A., et al. (2021). New creatinine- and cystatin C–based equations to estimate GFR without race. *New England Journal of Medicine, 385*, 1737–1749.

Kidney Disease: Improving Global Outcomes (KDIGO) CKD Work Group. (2024). KDIGO 2024 clinical practice guideline for the evaluation and management of chronic kidney disease. *Kidney International, 105*(4S), S117–S314.

Molenaar, E. A., Van Ameijden, E. J. C., Grobbee, D. E., & Numans, M. E. (2007). Comparison of routine care self-reported and biometrical data on hypertension and diabetes: Results of the Utrecht Health Project. *European Journal of Public Health, 17*(2), 199–205.

Thakkinstian, A., Ingsathit, A., Chaiprasert, A., Rattanasiri, S., Sangthawan, P., Gojaseni, P., Kiattisunthorn, K., Ongaj, L., & Anothaisintawee, T. (2011). A simplified clinical prediction score of chronic kidney disease: A cross-sectional-survey study. *BMC Nephrology, 12*, 45. https://doi.org/10.1186/1471-2369-12-45

Tran, H. T. B., et al. (2017). A simple questionnaire to detect chronic kidney disease patients from Long An province screening data in Vietnam. *BMC Research Notes, 10*, 523. https://doi.org/10.1186/s13104-017-2847-7

Vickers, A. J., & Elkin, E. B. (2006). Decision curve analysis: A novel method for evaluating prediction models. *Medical Decision Making, 26*(6), 565–574.

WHO Expert Consultation. (2004). Appropriate body-mass index for Asian populations and its implications for policy and intervention strategies. *The Lancet, 363*(9403), 157–163.